# Part 1: Data Preprocessing & Feature Engineering 

- In this section, the cleaned NYC taxi dataset from Assignment 1 is prepared for machine learning. New temporal, trip, and fare-based features are created to capture when trips occur, their characteristics, and cost structure. Pickup and dropoff boroughs are obtained using the taxi zone lookup table and encoded for use in machine learning models.

- Two target variables are created: `tip_amount` for the regression task and `high_tip`, a binary variable indicating whether the tip exceeds 20% of the fare amount for the classification task.

- Finally, the dataset is split into training (70%), validation (15%), and test (15%) sets using stratified sampling based on `high_tip`. Numeric features are scaled using `StandardScaler`, fitted only on the training data to prevent data leakage, and a summary of the final modeling features is provided.

## 1. Feature Engineering

### Loading data and filtering credit card trips 


In [1]:
import pandas as pd
import numpy as np

# Loading cleaned dataset from Assignment 1
df = pd.read_parquet("data/taxi_cleaned_features.parquet")

# Keeping only credit card payments 
df = df[df["payment_type"] == 1].copy()

print(df.shape)
df.head()

(2298412, 23)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,trip_duration_minutes,trip_speed_mph,pickup_hour,pickup_day_of_week
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1.0,1.8,1.0,N,140,236,1,...,3.75,0.0,1.0,18.75,2.5,0.0,6.600000,16.363636,0,Monday
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1.0,4.7,1.0,N,236,79,1,...,3.00,0.0,1.0,31.30,2.5,0.0,17.916667,15.739535,0,Monday
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1.0,1.4,1.0,N,79,211,1,...,2.00,0.0,1.0,17.00,2.5,0.0,8.300000,10.120482,0,Monday
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1.0,0.8,1.0,N,211,148,1,...,3.20,0.0,1.0,16.10,2.5,0.0,6.100000,7.868852,0,Monday
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1.0,4.7,1.0,N,148,141,1,...,6.90,0.0,1.0,41.50,2.5,0.0,32.383333,8.708183,0,Monday


### Load taxi zone lookup table

In [2]:
# Load taxi zone lookup table
zones = pd.read_csv("data/taxi_zone_lookup.csv")

zones.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### Creating the required engineered features

In [3]:
# Ensuring datetime columns are proper datetime types
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])


# a) Temporal features

df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.dayofweek   # Monday=0
df["is_weekend"] = df["pickup_day_of_week"].isin([5, 6])


# b) Trip features

df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# Avoid division by zero for speed calculation
df["trip_speed_mph"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["trip_distance"] / (df["trip_duration_minutes"] / 60),
    np.nan
)

df["log_trip_distance"] = np.log1p(df["trip_distance"])


# c) Fare features

df["fare_per_mile"] = np.where(
    df["trip_distance"] > 0,
    df["fare_amount"] / df["trip_distance"],
    np.nan
)

df["fare_per_minute"] = np.where(
    df["trip_duration_minutes"] > 0,
    df["fare_amount"] / df["trip_duration_minutes"],
    np.nan
)

df[[
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "trip_duration_minutes",
    "trip_speed_mph",
    "log_trip_distance",
    "fare_per_mile",
    "fare_per_minute"
]].head()

,pickup_hour,pickup_day_of_week,is_weekend,trip_duration_minutes,trip_speed_mph,log_trip_distance,fare_per_mile,fare_per_minute
1,0,0,False,6.600000,16.363636,1.029619,5.555556,1.515152
2,0,0,False,17.916667,15.739535,1.740466,4.957447,1.300465
3,0,0,False,8.300000,10.120482,0.875469,7.142857,1.204819
4,0,0,False,6.100000,7.868852,0.587787,9.875000,1.295082
5,0,0,False,32.383333,8.708183,1.740466,6.297872,0.914050


### Adding pickup and dropoff borough features from the lookup table

- The taxi zone lookup table was merged with the dataset to obtain the pickup and dropoff borough for each trip using the location IDs. These borough features will later be encoded for use in machine learning models.

In [4]:
# d) Zone features

# Keeping only the lookup fields that are necessary
zone_lookup = zones[["LocationID", "Borough"]].copy()

# Merge pickup borough
pickup_lookup = zone_lookup.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough"
})

df = df.merge(pickup_lookup, on="PULocationID", how="left")

# Merge dropoff borough
dropoff_lookup = zone_lookup.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough"
})

df = df.merge(dropoff_lookup, on="DOLocationID", how="left")

df[["PULocationID", "pickup_borough", "DOLocationID", "dropoff_borough"]].head()

,PULocationID,pickup_borough,DOLocationID,dropoff_borough
0,140,Manhattan,236,Manhattan
1,236,Manhattan,79,Manhattan
2,79,Manhattan,211,Manhattan
3,211,Manhattan,148,Manhattan
4,148,Manhattan,141,Manhattan


In [6]:
# Data quality checks
df.replace([np.inf, -np.inf], np.nan, inplace=True)

print(df[[
    "trip_speed_mph",
    "fare_per_minute",
    "pickup_borough",
    "dropoff_borough"
]].isnull().sum())

trip_speed_mph       33
fare_per_minute      33
pickup_borough      311
dropoff_borough    7850
dtype: int64


### Cleaning the data

In [7]:
df.dropna(subset=[
    "trip_speed_mph",
    "fare_per_minute",
    "pickup_borough",
    "dropoff_borough"
], inplace=True)

Rows containing missing values in engineered features (trip speed, fare per minute, or borough information) were removed to ensure the dataset is suitable for model training.

### One-Hot Encoding Borough Features

The pickup and dropoff borough columns are categorical variables. These features are converted into numerical format using one-hot encoding so they can be used by machine learning models.

In [8]:
df = pd.get_dummies(
    df,
    columns=["pickup_borough", "dropoff_borough"],
    prefix=["pickup", "dropoff"]
)

In [26]:
# Convert store_and_fwd_flag from categorical (Y/N) to numeric (1/0)
df["store_and_fwd_flag"] = df["store_and_fwd_flag"].map({"Y": 1, "N": 0})

## 2. Target Variable Creation 

- Two target variables are defined for the modeling tasks. `tip_amount` is used as the continuous target for the regression problem. A second binary target, `high_tip`, is created to indicate whether the tip exceeds 20% of the fare amount, which will be used for the classification task.

In [27]:
# Regression target 
y_reg = df["tip_amount"]

# Classification target, tip greater than 20% of fare
df["high_tip"] = (df["tip_amount"] > 0.20 * df["fare_amount"]).astype(int)

# Preview the new target column
df[["tip_amount", "fare_amount", "high_tip"]].head()

,tip_amount,fare_amount,high_tip
0,3.75,10.0,1
1,3.00,23.3,0
2,2.00,10.0,0
3,3.20,7.9,1
4,6.90,29.6,1


## 3. Data Splitting & Scaling

- The dataset is split into training (70%), validation (15%), and test (15%) sets. Stratified sampling is used based on the `high_tip` classification target to maintain the same class distribution across splits. Numeric features are scaled using `StandardScaler`, which is fitted only on the training data to prevent data leakage. Finally, the number of samples and class distribution for each split are reported, along with a summary of the modeling features.

### Selecting features and targets

In [28]:
from sklearn.model_selection import train_test_split

# Targets
y_reg = df["tip_amount"]
y_clf = df["high_tip"]

# Features - exclude targets
X = df.drop(columns=["tip_amount", "high_tip"])

In [17]:
# Columns to exclude from features including targets and raw datetime
excluded_features = [
    "tip_amount",              
    "high_tip",               
    "tpep_pickup_datetime",    
    "tpep_dropoff_datetime"    
]
X = df.drop(columns=excluded_features)

y_reg = df["tip_amount"]
y_clf = df["high_tip"]

Raw datetime columns (`tpep_pickup_datetime` and `tpep_dropoff_datetime`) were excluded from modeling because temporal information was already captured through engineered features such as `pickup_hour`, `pickup_day_of_week`, and `trip_duration_minutes`. The target variables `tip_amount` and `high_tip` were also excluded from the feature set to prevent data leakage.

### Train / validation / test split

In [29]:
# First split - 70% train, 30% temp

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_clf,
    test_size=0.30,
    stratify=y_clf,
    random_state=42
)

# Second split - 15% validation, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

### Documenting number of samples

In [30]:
print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 1603286
Validation size: 343561
Test size: 343562


### Checking class distribution

In [32]:
print("Train high_tip distribution:")
print(y_train.value_counts(normalize=True))

print("\nValidation high_tip distribution:")
print(y_val.value_counts(normalize=True))

print("\nTest high_tip distribution:")
print(y_test.value_counts(normalize=True))

Train high_tip distribution:
high_tip
1    0.760076
0    0.239924
Name: proportion, dtype: float64

Validation high_tip distribution:
high_tip
1    0.760078
0    0.239922
Name: proportion, dtype: float64

Test high_tip distribution:
high_tip
1    0.760075
0    0.239925
Name: proportion, dtype: float64


The dataset was split into training (70%), validation (15%), and test (15%) sets using stratified sampling based on the `high_tip` target. The class distribution remains consistent across all splits, with approximately 76% high-tip trips and 24% normal-tip trips, confirming that stratification was successful.

### Scaling numeric features

- Numeric features are standardized using `StandardScaler` so that variables with different ranges do not disproportionately influence the machine learning models. The scaler is fitted only on the training data and then applied to the validation and test sets to prevent data leakage.

In [33]:
from sklearn.preprocessing import StandardScaler

# Select only numeric columns
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns

scaler = StandardScaler()

# Keep dataframe structure
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

# Scale numeric columns only
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_val_scaled[numeric_cols] = scaler.transform(X_val[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

### Feature summary 

- The final feature set includes engineered temporal, trip, fare, and encoded borough features. The target variables and raw datetime columns are excluded from the model inputs to prevent data leakage and because their information is already captured by engineered features.

In [35]:
# Create a feature summary table
feature_summary = pd.DataFrame({
    "feature_name": X.columns,
    "data_type": X.dtypes.astype(str).values
})

print("Number of features used for modeling:", len(X.columns))
display(feature_summary)

# Excluded features with reasons
excluded_summary = pd.DataFrame({
    "excluded_feature": [
        "tip_amount",
        "high_tip",
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime"
    ],
    "reason": [
        "Regression target variable",
        "Classification target variable",
        "Raw datetime column; temporal information already captured by engineered features",
        "Raw datetime column; trip duration already engineered"
    ]
})

display(excluded_summary)

Number of features used for modeling: 40


,feature_name,data_type
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[ns]
2,tpep_dropoff_datetime,datetime64[ns]
3,passenger_count,float64
4,trip_distance,float64
5,RatecodeID,float64
6,store_and_fwd_flag,int64
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


,excluded_feature,reason
0,tip_amount,Regression target variable
1,high_tip,Classification target variable
2,tpep_pickup_datetime,Raw datetime column; temporal information alre...
3,tpep_dropoff_datetime,Raw datetime column; trip duration already eng...


The table above shows the final features used for modeling and their data types. These include engineered temporal, trip, and fare features as well as one-hot encoded borough features. Target variables and raw datetime columns were excluded from the feature set to prevent data leakage.